# 🏗️ Notebook 1 — Stock Exchange: Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Then **select the `.venv` kernel** in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses only the Python standard library + `pydantic` — nothing else to install, no Docker required.


## What is a stock exchange?

A **stock exchange** is a marketplace where buyers and sellers of a stock meet. It does **one** job extremely well: take orders, **match** buyers with sellers in a fair order, and publish the resulting trades + prices to the world.

> 💡 An exchange is **not** a brokerage. Your Robinhood/Fidelity account is a *brokerage* — it routes your orders to an exchange like **NYSE** or **Nasdaq**, which is where the actual matching happens. This lab is about that matching layer.

Every trading day, the world's exchanges match **tens of billions** of orders. The core of each is a small piece of code called the **matching engine**. We will build a tiny one by the end of Notebook 3.


## Functional requirements (what the system does)

- **Place** a limit order: side (`buy`/`sell`), price, quantity, symbol.
- **Cancel / modify** an open order.
- **Match** buys and sells when prices cross (a buy ≥ the lowest sell).
- **Publish** every trade and every change in the top of the book to subscribers.

## Non-functional requirements (how well it does it)

- **Determinism** — same input sequence ⇒ *identical* output. This is what makes replay, audit and dual-run testing possible.
- **Low latency** — microseconds inside the matcher, ~1 ms end-to-end.
- **Durability** — every order **persisted to disk before it is acknowledged** (a Write-Ahead Log, or WAL).
- **Fairness** — price-time priority. We'll dig into this in Notebook 3.


## Back-of-the-envelope

Let's compute a rough capacity target for a medium-size exchange. Change the inputs and see what falls out.

In [1]:
# Rough sizing exercise. Tweak the numbers and re-run.
symbols            = 10_000       # tradable instruments
peak_orders_per_s  = 1_000_000    # worldwide peak (Nasdaq peaks >10M msgs/s)
avg_order_bytes    = 80           # compact binary on the wire
subscribers        = 100_000      # market-data consumers
trades_per_s       = peak_orders_per_s // 5   # ~1 trade per 5 order events

print(f"Ingress bandwidth : {peak_orders_per_s * avg_order_bytes / 1e6:,.1f} MB/s")
print(f"Fan-out bandwidth : {trades_per_s * avg_order_bytes * subscribers / 1e9:,.1f} GB/s")
print(f"Daily order volume: {peak_orders_per_s * 3600 * 6.5 / 1e9:,.1f} B orders (6.5h session)")
print()
print("Key insight: fan-out dwarfs ingress. Market-data distribution is a ")
print("bandwidth + network problem, while matching is a CPU + memory problem.")

Ingress bandwidth : 80.0 MB/s
Fan-out bandwidth : 1,600.0 GB/s
Daily order volume: 23.4 B orders (6.5h session)

Key insight: fan-out dwarfs ingress. Market-data distribution is a 
bandwidth + network problem, while matching is a CPU + memory problem.


## High-level architecture

```
      [Trader] ──FIX / WebSocket──▶ ┌──────────────┐
                                    │   Gateway    │  auth, rate limit, normalize
                                    └──────┬───────┘
                                           │ one stream per symbol
                                           ▼
                                    ┌──────────────┐        ┌──────────────┐
                                    │  Sequencer   │───────▶│  WAL (disk)  │
                                    └──────┬───────┘        └──────────────┘
                                           │ monotonic sequence #
                                           ▼
                                    ┌──────────────┐        ┌──────────────┐
                                    │  Matching    │───────▶│ Trade feed   │──▶ subscribers
                                    │  engine      │        └──────────────┘
                                    └──────────────┘
```

Each box has **one** responsibility:

| Component     | Job                                                   | Scale strategy                |
|---------------|-------------------------------------------------------|-------------------------------|
| Gateway       | Terminate client sockets, auth, rate-limit            | Stateless → horizontal        |
| Sequencer     | Impose a total order on messages for a symbol         | One per symbol (or shard)     |
| WAL           | Durable record of every input **before** it matches   | Append-only log, fsync'd      |
| Matching engine | Own the order book, compute trades                 | **Single-threaded per symbol**|
| Trade feed    | Fan trades out to thousands of subscribers            | Pub/sub, many replicas        |

> 🔑 **Why single-threaded?** Determinism. If two threads can reorder messages, the same inputs can yield different trades. We'll demonstrate this in the next cell.


## Bad → Best: why determinism forces single-threaded matching

**Bad idea**: let a web framework spin up a thread per request and have each thread mutate the order book. It's fast to write, but the output depends on OS scheduling — run the same test twice and get different trades.

**Better idea**: place **one** sequencer in front of **one** matcher. All writes go through a single queue. The matcher is a plain `while True: process(queue.get())` loop.

The cell below shows both, side by side. Run it a few times — the multi-threaded version jitters; the single-threaded one is rock-solid.

In [2]:
import threading, queue, random, time
from collections import Counter

# --- BAD: many worker threads racing on a shared list ---------------------
def bad_run(orders):
    trades = []
    barrier = threading.Barrier(len(orders))   # all threads start together
    def worker(o):
        barrier.wait()
        # read → tiny random delay (context switch) → append:
        # without a lock, the final order depends on OS scheduling
        time.sleep(random.random() * 0.001)
        trades.append(o)
    ts = [threading.Thread(target=worker, args=(o,)) for o in orders]
    for t in ts: t.start()
    for t in ts: t.join()
    return tuple(trades)

# --- BEST: one queue, one matcher thread ----------------------------------
def best_run(orders):
    trades = []
    q = queue.Queue()
    for o in orders: q.put(o)
    q.put(None)
    def matcher():
        while True:
            o = q.get()
            if o is None: return
            trades.append(o)
    t = threading.Thread(target=matcher); t.start(); t.join()
    return tuple(trades)

orders = [101, 102, 103, 104, 105]

bad_results  = Counter(bad_run(orders)  for _ in range(30))
best_results = Counter(best_run(orders) for _ in range(30))

print(f"BAD  — distinct trade orderings across 30 runs: {len(bad_results)}")
print(f"        most common: {bad_results.most_common(1)[0][0]}")
print(f"BEST — distinct trade orderings across 30 runs: {len(best_results)}   ← deterministic!")
print(f"        always:      {next(iter(best_results))}")

BAD  — distinct trade orderings across 30 runs: 25
        most common: (104, 102, 101, 103, 105)
BEST — distinct trade orderings across 30 runs: 1   ← deterministic!
        always:      (101, 102, 103, 104, 105)


## Real-world sightings

| Exchange / system | Matching design                                                                 |
|-------------------|---------------------------------------------------------------------------------|
| **Nasdaq INET**   | Single-threaded matcher per symbol, C++, shared-memory IPC.                     |
| **NYSE Pillar**   | One matcher per symbol, replicated state machine for HA.                        |
| **LMAX Disruptor**| Famous pattern: single thread + lock-free ring buffer → 6M msgs/s on one core.  |
| **CME iLink**     | Sequencer + deterministic matcher, replay-from-WAL for disaster recovery.       |

Every one of them looks like the diagram above. Different language, same shape.

## Out of scope for this lab (so we stay small)

- Clearing & settlement (T+1), margin, risk checks.
- Regulatory reporting (CAT, MiFID II).
- Co-location, FPGA acceleration, kernel-bypass networking.
- Market microstructure (hidden orders, auctions, circuit breakers) — we'll touch circuit breakers briefly in Notebook 3.

Next up: **Notebook 2** — the concrete data model and public API.
